# Create gematria Text-Fabric features for BHSA

Tony Jurg · version 0.2.0 · 2026-10-02

Workflow adapted from [Create_TF-IDF_Text-Fabric_features](https://github.com/tonyjurg/Create_TF-IDF_Text-Fabric_features) (CC BY 4.0). See README and license notices. Run from this repository root after installing requirements.txt. This notebook generates a small sample; the local script creates the full dataset.


Clickable contents and section return links follow the pattern in Tony Jurg’s [BHSA TF-IDF notebook](https://github.com/tonyjurg/Create_TF-IDF_features_BHSA_TF/blob/main/create_tfidf_features.ipynb).

## Table of contents (ToC) <a class="anchor" id="TOC"></a>

* <a href="#bullet1">1 — Setup</a>
* <a href="#bullet2">2 — Inspect BHSA representations</a>
* <a href="#bullet3">3 — Calculate independently of Text-Fabric</a>
* <a href="#bullet4">4 — Generate and export a sample</a>
* <a href="#bullet5">5 — Reload and verify</a>
* <a href="#bullet6">6 — Attribution</a>

## 1 — Setup <a class="anchor" id="bullet1"></a>

[Back to ToC](#TOC)

Text-Fabric downloads the pinned BHSA snapshot into its standard data location (`~/text-fabric-data/github/etcbc/bhsa/tf/2021`). Required source files are verified by SHA-256; the project uses no separate BHSA cache.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from gematria_tf.bhsa import fetch_source, load_source
from gematria_tf.generator import generate
from gematria_tf import calculate

source = fetch_source()
api, lock = load_source(source)
print(lock["version"], lock["commit"])

## 2 — Inspect BHSA representations <a class="anchor" id="bullet2"></a>

[Back to ToC](#TOC)

The first two slots are the attached prefix and the remainder of בראשית. The full form joins them; maqaf and other nonempty trailers terminate a group.

In [ ]:
from gematria_tf.representations import read_words, represent
words = read_words(api)
texts, starts, ends = represent(words, "ketiv")
[(n, texts["lexeme"][n], texts["word"][n], texts["full"][n], starts[n], ends[n]) for n in range(1, 12)]

## 3 — Calculate independently of Text-Fabric <a class="anchor" id="bullet3"></a>

[Back to ToC](#TOC)

Methods and transformations are separate dimensions.

In [ ]:
assert calculate("חי") == 18
assert calculate("שלום", "gadol") == 936
assert calculate("בבל", transform="atbash") == 620
{method: calculate("שלום", method) for method in ("hechrechi", "gadol", "sidduri", "katan", "katan_mispari")}

## 4 — Generate and export a sample <a class="anchor" id="bullet4"></a>

[Back to ToC](#TOC)

A new output directory makes this cell safe to rerun. Both recorded readings are included. For the complete dataset use `python create_gematria_features.py --fetch --readings ketiv qere --output tf/2021` in a terminal, with an empty target directory.

In [ ]:
Path("output").mkdir(exist_ok=True)
output = Path(mkdtemp(prefix="notebook-", dir="output")).resolve()
manifest = generate(api, source, lock, output, readings=("ketiv", "qere"), limit=100)
print(output)
print(manifest["wordCount"], len(manifest["features"]), manifest["roundTripVerified"])

## 5 — Reload and verify <a class="anchor" id="bullet5"></a>

[Back to ToC](#TOC)

The exporter has already verified every emitted node/value. This cell shows the public Text-Fabric access pattern. Count full groups only at their start nodes.

In [ ]:
from tf.fabric import Fabric
TF = Fabric(locations=[str(source), str(output)], silent="deep")
check = TF.load("gem_hechrechi_full_ketiv_ident gem_full_start_ketiv", silent="deep")
assert check.F.gem_hechrechi_full_ketiv_ident.v(1) == 913
assert check.F.gem_hechrechi_full_ketiv_ident.v(2) == 913
assert check.F.gem_full_start_ketiv.v(2) == 1
print("Verified בראשית = 913 on both constituent slots")

## 6 — Attribution <a class="anchor" id="bullet6"></a>

[Back to ToC](#TOC)

BHSA: ETCBC, [DOI 10.17026/dans-z6y-skyh](https://doi.org/10.17026/dans-z6y-skyh), CC BY-NC 4.0. Notebook text/workflow: CC BY 4.0. New Python code: MIT. Numerical conventions and upstream implementation differences are documented in README.md and THIRD_PARTY_NOTICES.md.